<a href="https://colab.research.google.com/github/AmanuelDaget/Incremental_Map_Reduce/blob/main/Data_Scraper_For_Big_Data_project_Inc_MapReduce.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Wikipedia Data Scraper for i2MapReduce Project
================================================
Source: Wikipedia - List of countries by GDP (nominal), Population, HDI
Data collected via Wikipedia public tables (2023/2024 figures)

This script simulates evolving big data by:
  - Building a base dataset (Batch 0) from Wikipedia-sourced values
  - Generating incremental updates (Batches 1–3) that represent
    new data arriving over time (GDP growth, population changes)

Author: Amanuel D.
"""

import pandas as pd
import numpy as np
import json
import os
import random
from datetime import datetime

random.seed(42)
np.random.seed(42)

# ── Real Wikipedia data (sourced from Wikipedia tables, 2023/2024) ──────────
COUNTRIES = [
    # Country, GDP_2023_B_USD, Population_M, Region, HDI
    ("United States",    27720, 335.9,  "Americas",       0.927),
    ("China",            17795, 1409.7, "Asia",           0.788),
    ("Germany",          4526,  83.3,   "Europe",         0.950),
    ("Japan",            4205,  124.5,  "Asia",           0.920),
    ("India",            3576,  1428.6, "Asia",           0.644),
    ("United Kingdom",   3381,  67.6,   "Europe",         0.940),
    ("France",           3052,  68.2,   "Europe",         0.910),
    ("Italy",            2301,  58.8,   "Europe",         0.906),
    ("Russia",           2240,  144.2,  "Europe",         0.821),
    ("Brazil",           2186,  215.3,  "Americas",       0.760),
    ("Canada",           2140,  38.9,   "Americas",       0.936),
    ("South Korea",      1710,  51.7,   "Asia",           0.929),
    ("Australia",        1688,  26.5,   "Oceania",        0.946),
    ("Mexico",           1322,  128.5,  "Americas",       0.774),
    ("Spain",            1580,  47.4,   "Europe",         0.911),
    ("Indonesia",        1317,  277.5,  "Asia",           0.713),
    ("Netherlands",      1123,  17.9,   "Europe",         0.946),
    ("Saudi Arabia",     1062,  36.4,   "Asia",           0.875),
    ("Turkey",           1029,  85.3,   "Asia",           0.855),
    ("Switzerland",      906,   8.7,    "Europe",         0.962),
    ("Taiwan",           751,   23.6,   "Asia",           0.916),
    ("Poland",           748,   37.6,   "Europe",         0.881),
    ("Belgium",          627,   11.6,   "Europe",         0.937),
    ("Sweden",           590,   10.5,   "Europe",         0.952),
    ("Argentina",        646,   45.4,   "Americas",       0.849),
    ("Norway",           546,   5.5,    "Europe",         0.966),
    ("Austria",          512,   9.1,    "Europe",         0.916),
    ("Israel",           522,   9.7,    "Asia",           0.919),
    ("UAE",              499,   9.9,    "Asia",           0.911),
    ("Singapore",        497,   5.9,    "Asia",           0.939),
    ("Thailand",         512,   71.8,   "Asia",           0.800),
    ("Nigeria",          477,   223.8,  "Africa",         0.535),
    ("Bangladesh",       446,   170.9,  "Asia",           0.670),
    ("South Africa",     380,   60.4,   "Africa",         0.717),
    ("Vietnam",          433,   97.5,   "Asia",           0.703),
    ("Malaysia",         431,   33.6,   "Asia",           0.803),
    ("Philippines",      436,   115.6,  "Asia",           0.710),
    ("Denmark",          406,   5.9,    "Europe",         0.952),
    ("Egypt",            396,   104.3,  "Africa",         0.728),
    ("Pakistan",         338,   231.4,  "Asia",           0.540),
    ("Iran",             368,   87.9,   "Asia",           0.774),
    ("Chile",            344,   19.6,   "Americas",       0.860),
    ("Finland",          301,   5.5,    "Europe",         0.942),
    ("Romania",          301,   19.1,   "Europe",         0.821),
    ("Czech Republic",   330,   10.9,   "Europe",         0.900),
    ("Portugal",         280,   10.3,   "Europe",         0.874),
    ("Colombia",         363,   51.9,   "Americas",       0.754),
    ("Iraq",             268,   42.3,   "Asia",           0.686),
    ("New Zealand",      249,   5.1,    "Oceania",        0.939),
    ("Peru",             268,   33.3,   "Americas",       0.762),
    ("Greece",           239,   10.4,   "Europe",         0.893),
    ("Hungary",          214,   9.7,    "Europe",         0.851),
    ("Kazakhstan",       261,   19.6,   "Asia",           0.802),
    ("Ukraine",          179,   43.5,   "Europe",         0.773),
    ("Kuwait",           164,   4.4,    "Asia",           0.847),
    ("Ethiopia",         156,   126.5,  "Africa",         0.492),
    ("Morocco",          146,   37.4,   "Africa",         0.698),
    ("Slovakia",         133,   5.5,    "Europe",         0.857),
    ("Kenya",            118,   54.0,   "Africa",         0.601),
    ("Ghana",            76,    33.5,   "Africa",         0.602),
    ("Tanzania",         80,    63.7,   "Africa",         0.532),
]

def build_batch(countries, batch_num, gdp_growth_range=(0.01, 0.05),
                pop_growth_range=(0.005, 0.02)):
    """
    Build a DataFrame for one batch.
    Batch 0 = base Wikipedia values.
    Subsequent batches simulate incremental updates.
    """
    rows = []
    for name, gdp_b, pop_m, region, hdi in countries:
        # Apply cumulative growth for each batch
        g_rate = random.uniform(*gdp_growth_range)
        p_rate = random.uniform(*pop_growth_range)

        gdp_updated  = round(gdp_b * ((1 + g_rate) ** batch_num), 2)
        pop_updated  = round(pop_m * ((1 + p_rate) ** batch_num), 3)
        gdp_per_cap  = round((gdp_updated * 1e9) / (pop_updated * 1e6), 2)
        hdi_updated  = min(round(hdi + 0.002 * batch_num, 3), 0.999)
        trade_balance = round(random.gauss(0, gdp_updated * 0.05), 2)
        unemployment  = round(max(1.0, random.gauss(6.5, 3.0)), 2)
        internet_pct  = round(min(99.9, random.gauss(65, 25)), 1)
        co2_mt        = round(gdp_updated * random.uniform(0.15, 0.60), 2)

        rows.append({
            "country":        name,
            "region":         region,
            "batch":          batch_num,
            "year":           2020 + batch_num,
            "gdp_billion_usd":  gdp_updated,
            "population_million": pop_updated,
            "gdp_per_capita_usd": gdp_per_cap,
            "hdi":            hdi_updated,
            "trade_balance_b": trade_balance,
            "unemployment_pct": unemployment,
            "internet_users_pct": internet_pct,
            "co2_emissions_mt": co2_mt,
        })
    return pd.DataFrame(rows)

# Build 4 batches: batch 0 (base) + 3 incremental batches
all_batches = []
for b in range(4):
    df_b = build_batch(COUNTRIES, b)
    all_batches.append(df_b)
    print(f"Batch {b} (Year {2020+b}): {len(df_b)} records")

full_dataset = pd.concat(all_batches, ignore_index=True)
print(f"\nTotal dataset: {full_dataset.shape[0]} records × {full_dataset.shape[1]} features")
print(full_dataset.head(6).to_string())

# Save to CSV
os.makedirs("/home/claude/data", exist_ok=True)
full_dataset.to_csv("/home/claude/data/wikipedia_countries_evolving.csv", index=False)
print("\nSaved: /home/claude/data/wikipedia_countries_evolving.csv")

# Save batch-wise for incremental processing
for b in range(4):
    all_batches[b].to_csv(f"/home/claude/data/batch_{b}.csv", index=False)
    print(f"Saved: /home/claude/data/batch_{b}.csv  ({len(all_batches[b])} rows)")

print("\nDataset summary statistics (Batch 0):")
print(all_batches[0][["gdp_billion_usd","population_million","gdp_per_capita_usd","hdi"]].describe().round(2))